# Inference Multi-Character Aksara Lontara
## Fine-Tuned BLIP – Deteksi 1 sampai 4 Karakter per Gambar

### Penjelasan Pendekatan

Model BLIP yang telah dilatih pada gambar **tunggal** (1 karakter per gambar) sudah menunjukkan akurasi **100%**.
Notebook ini memanfaatkan model tersebut untuk mengenali **1 hingga 4 karakter** dalam satu gambar.

**Alur kerja:**

1. **Preprocessing** – Grayscale → Gaussian blur → adaptive threshold → morphological cleanup.
2. **Vertical Projection Segmentation** – Menghitung jumlah piksel hitam per kolom → menemukan gap (lembah) antar karakter → membelah gambar di titik gap. Pendekatan ini secara otomatis mengelompokkan titik/tanda dengan badan karakter di bawahnya karena berada di kolom yang sama.
3. **Bounding Box + Padding Proporsional** – Setiap region karakter di-crop dengan padding 15% dari ukuran region.
4. **Canvas Fixed 384×384 + Scaling** – Karakter di-scale agar mengisi ~55% canvas (konsisten dengan proporsi data training 1 karakter), lalu ditempatkan di tengah canvas putih 384×384.
5. **Inference Individual** – Setiap canvas di-caption oleh BLIP (identik dengan `inference_multi_character.ipynb`).
6. **Penggabungan Caption** – Hasil caption per karakter digabung menjadi satu deskripsi utuh.

**Mengapa Vertical Projection?**
Karakter Lontara sering memiliki **titik terpisah** di atas/bawah badan karakter. Contour detection biasa akan mendeteksi titik sebagai contour terpisah dan bisa salah merge. Vertical projection menghindari masalah ini karena titik dan badan karakter berada di kolom horizontal yang sama.

**Catatan Penting:**
- Aksara Lontara: **19 karakter dasar × 5 pola = 95 karakter** total.
- Pastikan karakter memiliki **jarak horizontal cukup** antar karakter.
- Maksimal **4 karakter** per gambar.
- Satu-satunya perbedaan dengan notebook legacy adalah **path model** (`model/blip/best_model`).

---
## Setup & Load Model

In [ ]:
import os
import re
import json
import cv2
import math
import torch
import numpy as np
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import matplotlib.pyplot as plt

# Mount Google Drive (khusus Colab)
from google.colab import drive
drive.mount('/content/drive')

# ─── Konfigurasi ───
DRIVE_DIR  = '/content/drive/MyDrive/ImageCaptioning'
MODEL_DIR  = f'{DRIVE_DIR}/model/blip/best_model'
TEST_DIR   = f'{DRIVE_DIR}/datatest'           # Folder gambar uji
MAX_LENGTH = 64
NUM_BEAMS  = 4
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device aktif: {DEVICE}')
print(f'Model path  : {MODEL_DIR}')
print(f'Test path   : {TEST_DIR}')

assert os.path.exists(MODEL_DIR), f'Model tidak ditemukan di {MODEL_DIR}! Pastikan hasil training sudah di-copy ke Drive.'

print(f'Memuat model dari: {MODEL_DIR}')
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

print('✅ Model BLIP siap digunakan!')

---
## Fungsi Segmentasi & Inference

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: FUNGSI SEGMENTASI & INFERENCE MULTI-KARAKTER (1–4)
# ═══════════════════════════════════════════════════════════════
# Vertical Projection Segmentation
# + Canvas fixed 384×384 dengan scaling proporsional (konsisten dgn data training)
# + Split di titik minimum projection (bukan midpoint gap)
# + Padding proporsional dengan BATAS AMAN (tidak melewati gap ke karakter sebelah)

MAX_CHARS    = 4
CANVAS_SIZE  = 384          # Ukuran canvas = resolusi input BLIP
FILL_RATIO   = 0.55         # Karakter mengisi ~55% canvas (mirip proporsi data training)

# ─── Preprocessing ───

def preprocess_binary(image_path):
    """Konversi gambar ke binary mask (karakter = putih, bg = hitam)."""
    img_cv = cv2.imread(image_path)
    if img_cv is None:
        raise ValueError(f'Gagal membaca gambar: {image_path}')

    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    thresh = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=25, C=10
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=1)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

    return img_cv, gray, thresh


# ─── Vertical Projection Segmentation ───

def find_split_points(projection, min_gap_width=3, valley_thresh=0):
    """Cari gap (kolom berturut-turut dengan projection <= valley_thresh)."""
    gaps = []
    in_gap = False
    gap_start = 0

    for i, val in enumerate(projection):
        if val <= valley_thresh:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_width = i - gap_start
                if gap_width >= min_gap_width:
                    gaps.append((gap_start, i, gap_width))
                in_gap = False

    if in_gap:
        gap_width = len(projection) - gap_start
        if gap_width >= min_gap_width:
            gaps.append((gap_start, len(projection), gap_width))

    return gaps


def segment_by_projection(image_path, min_gap_ratio=0.02, debug=False):
    """
    Segmentasi karakter menggunakan Vertical Projection Profile.
    - Adaptive valley threshold untuk gap yang tidak sepenuhnya 0.
    - Split di titik MINIMUM projection (bukan midpoint) → pemisahan lebih akurat.
    - Padding proporsional dengan BATAS AMAN: tidak melewati gap ke karakter sebelah.
    """
    img_cv, gray, thresh = preprocess_binary(image_path)
    h_img, w_img = gray.shape

    projection = np.sum(thresh > 0, axis=0)
    min_gap_px = max(2, int(w_img * min_gap_ratio))

    # Adaptive valley threshold: piksel noise di gap (5% dari max projection)
    max_proj = int(projection.max()) if len(projection) > 0 else 0
    valley_thresh = max(0, int(max_proj * 0.05))

    gaps = find_split_points(projection, min_gap_width=min_gap_px,
                             valley_thresh=valley_thresh)

    if debug:
        print(f'  Image size: {w_img}x{h_img}')
        print(f'  Min gap width: {min_gap_px}px, Valley thresh: {valley_thresh}')
        print(f'  Max projection: {max_proj}')
        print(f'  Gaps found: {len(gaps)}')
        for i, (gs, ge, gw) in enumerate(gaps):
            print(f'    Gap {i+1}: col {gs}–{ge} (width={gw})')

    content_cols = np.where(projection > valley_thresh)[0]
    if len(content_cols) == 0:
        return [], img_cv, thresh, projection

    col_start = content_cols[0]
    col_end = content_cols[-1] + 1

    inner_gaps = [(gs, ge, gw) for gs, ge, gw in gaps
                  if gs > col_start and ge < col_end]

    if debug:
        print(f'  Content range: col {col_start}–{col_end}')
        print(f'  Inner gaps (between chars): {len(inner_gaps)}')

    # Jika terlalu banyak gap, ambil yang terlebar (maks MAX_CHARS - 1)
    if len(inner_gaps) > MAX_CHARS - 1:
        inner_gaps = sorted(inner_gaps, key=lambda g: g[2], reverse=True)[:MAX_CHARS - 1]
        inner_gaps = sorted(inner_gaps, key=lambda g: g[0])

    split_cols = [col_start]
    for gs, ge, gw in inner_gaps:
        # Split di kolom dengan projection MINIMUM dalam gap (bukan midpoint)
        gap_segment = projection[gs:ge]
        min_idx = int(np.argmin(gap_segment))
        split_col = gs + min_idx
        split_cols.append(split_col)

        if debug:
            print(f'    Split at col {split_col} (min proj={projection[split_col]} in gap {gs}–{ge})')

    split_cols.append(col_end)

    # ─── Buat bounding box per region dengan SAFE PADDING ───
    raw_boxes = []
    num_regions = len(split_cols) - 1

    for r in range(num_regions):
        x_left = split_cols[r]
        x_right = split_cols[r + 1]

        region_strip = thresh[:, x_left:x_right]
        row_proj = np.sum(region_strip > 0, axis=1)
        active_rows = np.where(row_proj > 0)[0]

        if len(active_rows) == 0:
            continue

        y_top = active_rows[0]
        y_bot = active_rows[-1] + 1

        # Padding proporsional
        region_w = x_right - x_left
        region_h = y_bot - y_top
        pad_x = max(5, int(region_w * 0.20))
        pad_y = max(5, int(region_h * 0.20))

        # BATAS AMAN: padding x TIDAK boleh melewati gap ke konten karakter sebelah
        # Gap di kiri region r = inner_gaps[r-1], gap di kanan = inner_gaps[r]
        if r > 0:
            # Batas kiri = awal gap sebelumnya (di luar itu = konten karakter lain)
            safe_x_left = inner_gaps[r - 1][0]
        else:
            safe_x_left = 0

        if r < num_regions - 1:
            # Batas kanan = akhir gap sesudahnya (di luar itu = konten karakter lain)
            safe_x_right = inner_gaps[r][1]
        else:
            safe_x_right = w_img

        x1 = max(safe_x_left, x_left - pad_x)
        y1 = max(0, y_top - pad_y)
        x2 = min(safe_x_right, x_right + pad_x)
        y2 = min(h_img, y_bot + pad_y)

        if (x2 - x1) > 5 and (y2 - y1) > 5:
            raw_boxes.append((x1, y1, x2, y2))

    if debug:
        print(f'  → Karakter terdeteksi: {len(raw_boxes)}')
        for i, (x1, y1, x2, y2) in enumerate(raw_boxes):
            print(f'    Box {i+1}: x={x1}–{x2} (w={x2-x1}), y={y1}–{y2} (h={y2-y1})')

    return raw_boxes, img_cv, thresh, projection


# ─── Crop + Canvas fixed 384×384 dengan scaling ───

def crop_character_on_canvas(pil_img, box,
                              canvas_size=CANVAS_SIZE,
                              fill_ratio=FILL_RATIO):
    """
    Crop karakter lalu tempatkan di tengah canvas putih.
    Canvas = fixed 384×384 (resolusi input BLIP).
    Karakter di-scale agar sisi terpanjang ≈ fill_ratio × canvas_size.
    Ini memastikan proporsi karakter konsisten dengan data training (1 karakter/gambar).
    """
    x1, y1, x2, y2 = box
    cropped = pil_img.crop((x1, y1, x2, y2))
    cw, ch = cropped.size

    # Scale karakter: sisi terpanjang → fill_ratio * canvas_size
    max_dim = max(cw, ch)
    if max_dim > 0:
        target_dim = int(canvas_size * fill_ratio)
        scale_factor = target_dim / max_dim
        new_cw = max(1, int(cw * scale_factor))
        new_ch = max(1, int(ch * scale_factor))
        # Pastikan tidak melebihi canvas
        new_cw = min(new_cw, canvas_size - 4)
        new_ch = min(new_ch, canvas_size - 4)
        cropped = cropped.resize((new_cw, new_ch), Image.LANCZOS)
        cw, ch = new_cw, new_ch

    canvas = Image.new('RGB', (canvas_size, canvas_size), (255, 255, 255))
    paste_x = (canvas_size - cw) // 2
    paste_y = (canvas_size - ch) // 2
    canvas.paste(cropped, (paste_x, paste_y))

    return canvas


# ─── Inference (identik dengan inference_local.ipynb) ───

def predict_single(pil_img):
    """Captioning 1 gambar — IDENTIK dengan inference_local.ipynb."""
    inputs = processor(images=pil_img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=2
        )
    return processor.decode(output[0], skip_special_tokens=True)


# ─── Gabung Caption ───

def extract_char_name(caption):
    """Ekstrak nama karakter dari caption model."""
    patterns = [
        r'yaitu karakter\s+(\w+)',
        r'adalah\s+(\w+)\s+dengan',
        r'merupakan karakter\s+(\w+)',
        r'karakter\s+(\w+)\s+memiliki',
        r'karakter\s+(\w+)\s+dengan',
    ]
    for pat in patterns:
        m = re.search(pat, caption, re.IGNORECASE)
        if m:
            return m.group(1)
    return None


def combine_captions(captions):
    """Gabungkan caption individual (1–4 karakter)."""
    n = len(captions)
    if n == 0:
        return 'Tidak ada karakter yang terdeteksi.'
    if n == 1:
        return captions[0]

    names = [extract_char_name(c) for c in captions]
    valid_names = [nm for nm in names if nm is not None]

    if len(valid_names) == n:
        if n == 2:
            nama_str = f'{valid_names[0]} dan {valid_names[1]}'
        else:
            nama_str = ', '.join(valid_names[:-1]) + f', dan {valid_names[-1]}'
        intro = f'Gambar ini terdiri dari {n} karakter lontara yaitu karakter {nama_str}.'
    else:
        intro = f'Gambar ini terdiri dari {n} karakter lontara.'

    details = []
    for i, cap in enumerate(captions, 1):
        name = names[i-1] if names[i-1] else '?'
        clean = re.sub(r'^Gambar ini terdiri dari \d+ karakter lontara[^.]*\.?\s*', '', cap).strip()
        if not clean:
            clean = cap
        details.append(f'Karakter ke-{i} ({name}): {clean}')

    return intro + ' ' + ' '.join(details)


# ─── Entry Point ───

def predict_multi(image_path, debug=False):
    """
    Segmentasi → crop pada canvas fixed 384×384 → inference → gabung.
    """
    pil_img = Image.open(image_path).convert('RGB')

    boxes, img_cv, thresh, projection = segment_by_projection(
        image_path, min_gap_ratio=0.02, debug=debug
    )

    if len(boxes) <= 1:
        cap = predict_single(pil_img)
        return cap, [pil_img], [cap], [], projection

    char_imgs = []
    char_caps = []
    for box in boxes:
        cimg = crop_character_on_canvas(pil_img, box)
        char_imgs.append(cimg)
        char_caps.append(predict_single(cimg))

    combined = combine_captions(char_caps)
    return combined, char_imgs, char_caps, boxes, projection

print(f'✅ Segmentasi + Canvas fixed {CANVAS_SIZE}×{CANVAS_SIZE} (fill={FILL_RATIO}) siap!')
print(f'   Split di titik minimum projection.')
print(f'   Padding dengan BATAS AMAN → crop tidak mengandung bagian karakter sebelah.')

---
## Demo Inference (Satu Gambar)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: DEMO INFERENCE (SATU GAMBAR — 1 SAMPAI 4 KARAKTER)
# ═══════════════════════════════════════════════════════════════

# Ganti dengan nama file gambar uji Anda
TEST_IMAGE = f'{TEST_DIR}/sampel03.png'   # <-- sesuaikan nama file

if not os.path.exists(TEST_IMAGE):
    print(f'⚠️ Gambar tidak ditemukan: {TEST_IMAGE}')
    print('Pastikan Anda sudah meng-upload gambar ke folder datatest di Google Drive.')
else:
    combined, char_imgs, char_caps, boxes, projection = predict_multi(TEST_IMAGE, debug=True)

    n = len(char_imgs)

    print('=' * 60)
    print(f'Jumlah karakter terdeteksi: {n}')
    print('=' * 60)
    print('📌 CAPTION GABUNGAN:')
    print(combined)
    print('=' * 60)
    print('📋 DETAIL PER KARAKTER:')
    for i, cap in enumerate(char_caps, 1):
        print(f'  [{i}] {cap}')
    print('=' * 60)

    # ─── Visualisasi Debug: Region Berwarna + Projection ───
    img_cv_dbg, _, thresh_dbg = preprocess_binary(TEST_IMAGE)
    h_img, w_img = thresh_dbg.shape

    # Warna untuk setiap karakter (RGB)
    region_colors = [
        (255, 80, 80),    # Merah
        (80, 200, 80),    # Hijau
        (80, 120, 255),   # Biru
        (255, 180, 50),   # Oranye
    ]

    # Buat binary mask BERWARNA — setiap region karakter diberi warna berbeda
    colored_mask = np.ones((h_img, w_img, 3), dtype=np.uint8) * 30  # background gelap

    if len(boxes) > 0:
        for idx_b, (x1, y1, x2, y2) in enumerate(boxes):
            clr = region_colors[idx_b % len(region_colors)]
            region = thresh_dbg[y1:y2, x1:x2]
            for c in range(3):
                colored_mask[y1:y2, x1:x2, c] = np.where(
                    region > 0,
                    clr[c],
                    colored_mask[y1:y2, x1:x2, c]
                )
            cv2.rectangle(colored_mask, (x1, y1), (x2, y2), (255, 255, 255), 1)
            cv2.putText(colored_mask, str(idx_b + 1),
                        (x1 + 2, y1 + 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    else:
        colored_mask[thresh_dbg > 0] = [200, 200, 200]

    fig_dbg, axes_dbg = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Gambar asli + bounding box berwarna
    img_asli_draw = cv2.cvtColor(img_cv_dbg, cv2.COLOR_BGR2RGB).copy()
    for idx_b, (x1, y1, x2, y2) in enumerate(boxes):
        clr = region_colors[idx_b % len(region_colors)]
        cv2.rectangle(img_asli_draw, (x1, y1), (x2, y2), clr, 2)
        cv2.putText(img_asli_draw, f'Chr {idx_b+1}',
                    (x1, max(y1 - 5, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, clr, 1)
    axes_dbg[0].imshow(img_asli_draw)
    axes_dbg[0].set_title('Gambar Asli + Bounding Box', fontweight='bold')
    axes_dbg[0].axis('off')

    # Panel 2: Binary mask berwarna per region
    axes_dbg[1].imshow(colored_mask)
    axes_dbg[1].set_title('Penanda Region (warna = 1 karakter)', fontweight='bold')
    axes_dbg[1].axis('off')
    from matplotlib.patches import Patch
    legend_patches = []
    for idx_b in range(len(boxes)):
        clr_norm = tuple(c / 255.0 for c in region_colors[idx_b % len(region_colors)])
        legend_patches.append(Patch(facecolor=clr_norm, label=f'Karakter {idx_b+1}'))
    if legend_patches:
        axes_dbg[1].legend(handles=legend_patches, loc='upper right', fontsize=8)

    # Panel 3: Vertical projection + split lines berwarna
    axes_dbg[2].fill_between(range(len(projection)), projection, alpha=0.4, color='gray')
    axes_dbg[2].set_title('Vertical Projection + Split Points', fontweight='bold')
    axes_dbg[2].set_xlabel('Kolom (x)')
    axes_dbg[2].set_ylabel('Jumlah piksel hitam')
    for idx_b, (x1, y1, x2, y2) in enumerate(boxes):
        clr_norm = tuple(c / 255.0 for c in region_colors[idx_b % len(region_colors)])
        axes_dbg[2].fill_between(range(x1, x2), projection[x1:x2],
                                 alpha=0.5, color=clr_norm, label=f'Chr {idx_b+1}')
    for idx_b, (x1, y1, x2, y2) in enumerate(boxes):
        if idx_b > 0:
            split_x = (boxes[idx_b - 1][2] + x1) // 2
            axes_dbg[2].axvline(x=split_x, color='red', linestyle='-',
                                linewidth=2, alpha=0.8)
            axes_dbg[2].text(split_x, projection.max() * 0.95, '✂',
                             ha='center', fontsize=12)
    axes_dbg[2].legend(fontsize=8, loc='upper left')
    plt.suptitle(f'Debug Segmentasi — {n} karakter terdeteksi', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ─── Visualisasi Hasil Inference (format list rapi) ───
    if n == 1 and len(boxes) == 0:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(Image.open(TEST_IMAGE).convert('RGB'))
        axes[0].set_title('Gambar Input', fontsize=11, fontweight='bold')
        axes[0].axis('off')
        axes[1].text(0.5, 0.5, char_caps[0], wrap=True,
                     ha='center', va='center', fontsize=10,
                     transform=axes[1].transAxes)
        axes[1].set_title('Caption', fontsize=11, fontweight='bold')
        axes[1].axis('off')
    else:
        # Layout: baris atas = gambar (input + crops), baris bawah = caption list
        fig = plt.figure(figsize=(4 * (n + 1), 8))

        # --- Baris atas: Input + Crops ---
        gs = fig.add_gridspec(2, n + 1, height_ratios=[3, 2], hspace=0.3)

        # Input + Bounding Box
        ax_input = fig.add_subplot(gs[0, 0])
        img_input = Image.open(TEST_IMAGE).convert('RGB')
        img_draw = np.array(img_input).copy()
        for idx_b, box in enumerate(boxes):
            x1, y1, x2, y2 = box
            clr = region_colors[idx_b % len(region_colors)]
            cv2.rectangle(img_draw, (x1, y1), (x2, y2), clr, 3)
            cv2.putText(img_draw, str(idx_b + 1), (x1 + 3, y2 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, clr, 2)
        ax_input.imshow(img_draw)
        ax_input.set_title('Input + Bounding Box', fontsize=10, fontweight='bold')
        ax_input.axis('off')

        # Crops per karakter
        for i in range(n):
            ax_crop = fig.add_subplot(gs[0, i + 1])
            clr_norm = tuple(c / 255.0 for c in region_colors[i % len(region_colors)])
            ax_crop.imshow(char_imgs[i])
            ax_crop.set_title(f'Karakter {i + 1}', fontsize=10,
                              fontweight='bold', color=clr_norm)
            for spine in ax_crop.spines.values():
                spine.set_edgecolor(clr_norm)
                spine.set_linewidth(3)
                spine.set_visible(True)
            ax_crop.set_xticks([])
            ax_crop.set_yticks([])

        # --- Baris bawah: Caption dalam format LIST ---
        ax_caption = fig.add_subplot(gs[1, :])
        ax_caption.axis('off')

        # Susun caption sebagai list vertikal dengan penanda warna
        caption_lines = []
        caption_lines.append(f'Caption Gabungan:')
        caption_lines.append(f'  {combined}')
        caption_lines.append('')
        caption_lines.append('Detail per Karakter:')

        y_pos = 0.95
        line_height = 0.12 if n <= 2 else 0.10

        # Judul "Caption Gabungan"
        ax_caption.text(0.02, y_pos, 'Caption Gabungan:', fontsize=11,
                        fontweight='bold', va='top', transform=ax_caption.transAxes)
        y_pos -= line_height * 0.8

        # Teks gabungan (wrap manual jika panjang)
        ax_caption.text(0.02, y_pos, combined, fontsize=9, va='top',
                        transform=ax_caption.transAxes, wrap=True,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                                  edgecolor='gray', alpha=0.8))
        y_pos -= line_height * 1.8

        # Judul "Detail per Karakter"
        ax_caption.text(0.02, y_pos, 'Detail per Karakter:', fontsize=11,
                        fontweight='bold', va='top', transform=ax_caption.transAxes)
        y_pos -= line_height * 0.7

        # List caption per karakter dengan bullet warna
        for i, cap in enumerate(char_caps):
            clr_norm = tuple(c / 255.0 for c in region_colors[i % len(region_colors)])
            name = extract_char_name(cap) or '?'
            # Bullet berwarna + nomor
            ax_caption.text(0.02, y_pos, '●', fontsize=14, color=clr_norm,
                            va='top', transform=ax_caption.transAxes)
            ax_caption.text(0.05, y_pos, f'Karakter {i+1} ({name}): ', fontsize=10,
                            fontweight='bold', color=clr_norm, va='top',
                            transform=ax_caption.transAxes)
            # Caption text
            ax_caption.text(0.05, y_pos - line_height * 0.5, cap, fontsize=9,
                            va='top', transform=ax_caption.transAxes, wrap=True)
            y_pos -= line_height * 1.3

        plt.suptitle(f'Inference: {n} karakter terdeteksi', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

---
## Batch Inference (Seluruh Folder)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: BATCH INFERENCE (SELURUH FOLDER DATATEST)
# ═══════════════════════════════════════════════════════════════

VALID_EXT = ('.png', '.jpg', '.jpeg')

if not os.path.exists(TEST_DIR):
    print(f'⚠️ Folder {TEST_DIR} tidak ditemukan. Buat folder datatest di Google Drive Anda.')
else:
    images = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(VALID_EXT)])
    print(f'Ditemukan {len(images)} gambar di folder datatest\n')

    results = []
    for img_name in images:
        img_path = os.path.join(TEST_DIR, img_name)
        try:
            cap, char_imgs, char_caps, boxes, _ = predict_multi(img_path)
            n_chars = len(char_imgs)
            results.append({
                'gambar': img_name,
                'jumlah_karakter': n_chars,
                'caption_gabungan': cap,
                'caption_per_karakter': char_caps
            })
            print(f'{img_name} ({n_chars} char) → {cap}')
        except Exception as e:
            print(f'{img_name} → ERROR: {e}')

    # Simpan hasil ke Drive
    out_dir = f'{DRIVE_DIR}/hasil_inference_multi'
    os.makedirs(out_dir, exist_ok=True)
    out_file = f'{out_dir}/hasil_multi.json'
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'\n✅ Hasil disimpan di: {out_file}')

    # ─── Visualisasi grid (maks 12 gambar) ───
    show = results[:12]
    cols = min(4, len(show))
    rows = math.ceil(len(show) / cols) if cols > 0 else 1
    if len(show) > 0:
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
        if rows * cols == 1:
            axes = [axes]
        else:
            axes = np.array(axes).flatten()
        for idx, item in enumerate(show):
            img = Image.open(os.path.join(TEST_DIR, item['gambar']))
            axes[idx].imshow(img)
            title = f"{item['gambar']}\n({item['jumlah_karakter']} char)"
            axes[idx].set_title(title, fontsize=8)
            axes[idx].axis('off')
        for idx in range(len(show), len(axes)):
            axes[idx].axis('off')
        plt.tight_layout()
        plt.show()

    print(f'\n✅ Selesai! Total {len(results)} gambar diproses.')